<a href="https://colab.research.google.com/github/jk5868/CBOW---NLP-Embeddings/blob/main/Copy_of_Untitled.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Corpus definition
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "i love deep learning and natural language processing",
    "tensorflow is an open source machine learning framework"
]

# 2. Tokenization and vocabulary mapping
words = " ".join(corpus).split()
print(words)
vocab = sorted(list(set(words)))
print(vocab)
vocab_size = len(vocab)
print(vocab_size)

word_to_index = {word: i for i, word in enumerate(vocab)}
print(word_to_index)
index_to_word = {i: word for i, word in enumerate(vocab)}
print(index_to_word)

# print(f"Vocabulary Size: {vocab_size}")

['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', 'i', 'love', 'deep', 'learning', 'and', 'natural', 'language', 'processing', 'tensorflow', 'is', 'an', 'open', 'source', 'machine', 'learning', 'framework']
['an', 'and', 'brown', 'deep', 'dog', 'fox', 'framework', 'i', 'is', 'jumps', 'language', 'lazy', 'learning', 'love', 'machine', 'natural', 'open', 'over', 'processing', 'quick', 'source', 'tensorflow', 'the']
23
{'an': 0, 'and': 1, 'brown': 2, 'deep': 3, 'dog': 4, 'fox': 5, 'framework': 6, 'i': 7, 'is': 8, 'jumps': 9, 'language': 10, 'lazy': 11, 'learning': 12, 'love': 13, 'machine': 14, 'natural': 15, 'open': 16, 'over': 17, 'processing': 18, 'quick': 19, 'source': 20, 'tensorflow': 21, 'the': 22}
{0: 'an', 1: 'and', 2: 'brown', 3: 'deep', 4: 'dog', 5: 'fox', 6: 'framework', 7: 'i', 8: 'is', 9: 'jumps', 10: 'language', 11: 'lazy', 12: 'learning', 13: 'love', 14: 'machine', 15: 'natural', 16: 'open', 17: 'over', 18: 'processing', 19: 'quick', 20: 'source', 21:

In [ ]:
WINDOW_SIZE = 2

X_train = []
y_train = []

for sentence in corpus:
    tokenized = sentence.split()
    for i, target in enumerate(tokenized):
        # Determine the sliding window bounds
        start = max(0, i - WINDOW_SIZE)
        end = min(len(tokenized), i + WINDOW_SIZE + 1)

        # Gather context words (excluding the target itself)
        context = [tokenized[j] for j in range(start, end) if j != i]

        # Only keep full windows for simplicity
        if len(context) == WINDOW_SIZE * 2:
            context_indices = [word_to_index[w] for w in context]
            target_index = word_to_index[target]

            X_train.append(context_indices)
            y_train.append(target_index)

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"Training shapes -> X: {X_train.shape}, y: {y_train.shape}")
# Example item: X_train[0] (context IDs) maps to y_train[0] (target ID)


Training shapes -> X: (13, 4), y: (13,)


In [ ]:
EMBEDDING_DIM = 10

# Input accepts a fixed number of context words (WINDOW_SIZE * 2)
input_layer = layers.Input(shape=(WINDOW_SIZE * 2,), dtype=tf.int32, name="Context_Words")

# Embedding layer: maps token indices to dense vectors
embedding_layer = layers.Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM, name="Word_Embeddings")
embedded = embedding_layer(input_layer)

# Crucial CBOW step: Average the context vectors element-wise
mean_embedding = layers.GlobalAveragePooling1D(name="Mean_Context_Vector")(embedded)

# Output layer: Predict probabilities across the entire vocabulary
output_layer = layers.Dense(units=vocab_size, activation="softmax", name="Target_Word_Prediction")(mean_embedding)

# Compile the model
cbow_model = models.Model(inputs=input_layer, outputs=output_layer)
cbow_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cbow_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Context_Words (InputLayer)      │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Word_Embeddings (Embedding)     │ (None, 4, 10)          │           230 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Mean_Context_Vector             │ (None, 10)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Target_Word_Prediction (Dense)  │ (None, 23)             │           253 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 483 (1.89 KB)

 Trainable params: 483 (1.89 KB)

 Non-trainable params: 0 (0.00 B)